# 03 — Model Comparison (Phase 3)

Compares 19 surrogate model configurations across 4 families (Polynomial, Kriging, Neural Network, RBF).

**Experiments:** MC-1 (full comparison), MC-2 (hyperparameter sensitivity)

**Source:** `outputs/phase3/mc1_summary.csv`, `mc2_*.csv`, `decision_gate.txt`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

ROOT = Path('..').resolve()
P3 = ROOT / 'outputs' / 'phase3'
mc1 = pd.read_csv(P3 / 'mc1_summary.csv')

## Table MC1: Models ranked by RMSE on v_out_mean

In [ ]:
vout = mc1[mc1['output'] == 'v_out_mean'].sort_values('rmse')
print(vout[['config_id', 'rmse', 'r2', 'mae', 'max_ae']].to_string(index=False))

## Table MC2: NRMSE Heatmap

In [ ]:
outputs = ['v_out_mean', 'i_l_mean', 'v_out_ripple', 'i_l_ripple', 'efficiency']
pivot = mc1.pivot_table(index='config_id', columns='output', values='nrmse')[outputs]
# Sort by v_out_mean
pivot = pivot.sort_values('v_out_mean')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('MC-1: NRMSE Heatmap (models x outputs)')
plt.tight_layout()
plt.show()

## Table MC3: Training and Prediction Times

In [ ]:
timing = mc1[mc1['output'] == 'v_out_mean'][['config_id', 'train_s', 'pred_us']].sort_values('train_s')
print(timing.to_string(index=False))

## MC-2: Hyperparameter Sensitivity

In [ ]:
# Polynomial
poly = pd.read_csv(P3 / 'mc2_poly.csv')
poly_vout = poly[poly['output'] == 'v_out_mean']
print('Polynomial HP sweep (v_out_mean):')
print(poly_vout[['degree', 'regulariser', 'alpha', 'rmse', 'r2']].sort_values('rmse').head(10).to_string(index=False))

print()

# Neural Network
nn = pd.read_csv(P3 / 'mc2_nn.csv')
nn_vout = nn[nn['output'] == 'v_out_mean']
print('Neural Network HP sweep (v_out_mean):')
print(nn_vout[['config', 'hidden', 'activation', 'dropout', 'weight_decay', 'rmse']].sort_values('rmse').to_string(index=False))

In [ ]:
# GP kernel comparison (from MC-1)
gp = pd.read_csv(P3 / 'mc2_gp.csv')
gp_vout = gp[gp['output'] == 'v_out_mean']
print('GP kernel comparison (v_out_mean):')
print(gp_vout[['config_id', 'kernel', 'ARD', 'nugget', 'rmse', 'r2']].sort_values('rmse').to_string(index=False))

print()

# RBF basis function comparison
rbf = pd.read_csv(P3 / 'mc2_rbf.csv')
rbf_vout = rbf[rbf['output'] == 'v_out_mean']
print('RBF basis comparison (v_out_mean):')
print(rbf_vout[['config_id', 'basis_function', 'rmse', 'r2']].sort_values('rmse').to_string(index=False))

## Parity and Residual Plots

In [ ]:
from IPython.display import Image, display
for fig_name in ['mc1_parity_top3.png', 'mc1_residuals_top3.png', 'mc1_residuals_all.png']:
    p = P3 / 'figures' / fig_name
    if p.exists():
        print(f'\n{fig_name}:')
        display(Image(filename=str(p), width=800))

In [ ]:
print(open(P3 / 'decision_gate.txt').read())

## Conclusion

**Top 3:** GP-M52, GP-SE, GP-SE-noARD (all Kriging). Kriging dominates across all outputs.
GP-M52 achieves RMSE=0.546V, R²=0.9991 on v_out_mean. These proceed to Phase 4.